**Pedido 1: Concentração do saldo na carteira**

Preciso entender quanto do nosso dinheiro depende de poucos clientes. Considerando só os clientes com saldo positivo, divida a base em 10 grupos iguais, do maior saldo para o menor.

Para cada grupo, quero ver:

quantos clientes tem;
o saldo total;
quanto isso representa do saldo total do banco, em %;
o percentual acumulado até aquele grupo;
o saldo médio.

Quero uma linha por grupo, ordenada do maior para o menor saldo.

In [0]:
WITH cte AS(    
    SELECT
        *,
        ntile(10) OVER(ORDER BY Balance DESC) AS grupos

    FROM andre_testes.bank_customer_churn
    WHERE Balance > 0
),

cte2 AS(
    SELECT
        grupos,
        COUNT(CustomerId) AS qtd_clientes,
        SUM(Balance) AS saldo_total,
        SUM(Balance)/(SELECT SUM(Balance) FROM andre_testes.bank_customer_churn WHERE Balance > 0)*100 AS percent_do_total,
        ROUND(AVG(Balance), 2) AS saldo_medio

    FROM cte
    GROUP BY grupos
)

SELECT
    grupos,
    qtd_clientes,
    saldo_total,
    percent_do_total,
    SUM(percent_do_total) OVER(ORDER BY grupos) AS percent_acumulado,
    saldo_medio

FROM cte2

**Pedido 2: Clientes premium de cada país**

Quero uma lista dos 10 clientes com maior saldo em cada país, para a equipe comercial abordar pessoalmente.

Para cada cliente, mostre:

ID, país, idade e saldo;
a posição dele no ranking do próprio país;
quantas vezes o saldo dele é maior que o saldo médio dos clientes daquele país (considerando só quem tem saldo positivo);
a diferença de saldo para o cliente logo abaixo dele no ranking.

Se houver empate de saldo, quero que os empatados apareçam com a mesma posição. Organize por país e depois por posição.

In [0]:
WITH cte AS(
    SELECT
        CustomerId,
        Geography,
        Surname,
        Balance,
        dense_rank() OVER(
            PARTITION BY Geography
            ORDER BY Balance DESC
        ) AS rank,
        ROUND(Balance / AVG(CASE WHEN Balance > 0 THEN Balance END) OVER(PARTITION BY Geography), 2) AS vezes_maior_media,
        Balance - lead(Balance, 1) OVER(
            PARTITION BY Geography
            ORDER BY Balance DESC
        ) AS diferenca_abaixo

    FROM andre_testes.bank_customer_churn
)

SELECT * FROM cte

WHERE rank <= 10
ORDER BY Geography, rank


**Pedido 3: Maturidade da base por tempo de relacionamento**

Quero saber se clientes mais antigos são mais engajados com o banco. Agrupe os clientes por anos de relacionamento (0 a 10), cada ano como um grupo.

Para cada grupo, mostre:

o número de clientes;
o saldo médio (só clientes com saldo positivo);
o % de clientes com saldo zero;
o % de clientes com 2 ou mais produtos.

Para o saldo médio e o % com 2 ou mais produtos, quero também a variação em relação ao ano anterior de relacionamento, em pontos absolutos.

Uma linha por ano, em ordem crescente.

In [0]:
WITH cte AS(
    SELECT
        Tenure,
        COUNT(*) AS Clientes,
        ROUND(AVG(CASE WHEN Balance > 0 THEN Balance END), 2) AS Saldo_Medio,
        ROUND((COUNT(CASE WHEN Balance = 0 THEN 1 END) / COUNT(*))*100, 2) AS Percent_Clientes_Saldo_Zero,
        ROUND((COUNT(CASE WHEN NumOfProducts >= 2 THEN 1 END) / COUNT(*))*100, 2) AS Percent_Clientes_2Mais_Produtos

    FROM workspace.andre_testes.bank_customer_churn

    GROUP BY Tenure
    ORDER BY Tenure
)

SELECT
    Tenure,
    Clientes,
    Saldo_Medio,
    Saldo_Medio - lag(Saldo_Medio) OVER(ORDER BY Tenure) AS Diferenca_Saldo_Medio,
    Percent_Clientes_2Mais_Produtos,
    ROUND(Percent_Clientes_2Mais_Produtos - lag(Percent_Clientes_2Mais_Produtos) OVER(ORDER BY Tenure), 2) AS Diferenca_2Mais_Produtos,
    Percent_Clientes_Saldo_Zero

FROM cte

**Pedido 4: Matriz de segmentação por faixa etária e país**

Preciso de uma visão cruzada para decidir onde investir em campanhas. Crie as faixas etárias: 18 a 29, 30 a 39, 40 a 49, 50 a 59 e 60 ou mais. Cruze cada faixa com cada país.

Para cada combinação de faixa e país, quero ver:

o número de clientes e quanto isso representa da base total, em %;
o saldo médio entre os clientes com saldo positivo;
o % de clientes com saldo zero;
um índice de saldo médio em que 100 é a média geral do banco (acima de 100 está acima da média).

Quero que o resultado apareça em formato de matriz, com as faixas etárias nas linhas e os países nas colunas. Mostre o número de clientes e o índice de saldo lado a lado. Marque de alguma forma as combinações com menos de 100 clientes, porque não quero tomar decisão em cima de amostra pequena.

Quero também uma frase de conclusão: em qual combinação está o maior índice de saldo e em qual está o menor, desconsiderando as combinações marcadas como amostra pequena.

In [0]:
WITH cte AS(
    SELECT
        *,
        COUNT(*) OVER() AS Total_Clientes,
        CASE 
            WHEN Age BETWEEN 18 AND 29 THEN '18 a 29'
            WHEN Age BETWEEN 30 AND 39 THEN '30 a 39'
            WHEN Age BETWEEN 40 AND 49 THEN '40 a 49'
            WHEN Age BETWEEN 50 AND 59 THEN '50 a 59'
            WHEN Age >= 60             THEN '60+'
            ELSE '0'
        END AS Faixa_Etaria

    FROM workspace.andre_testes.bank_customer_churn
)

SELECT
    Geography,
    Faixa_Etaria,
    COUNT(*) AS Clientes,
    ROUND((COUNT(*) / MAX(Total_Clientes))*100,2) AS Pct_Base

FROM cte
GROUP BY Geography, Faixa_Etaria
ORDER BY Geography, Faixa_Etaria